# RAG Pipeline Design, Chunking Strategy & Retrieval Timing

### Source : KNOWLEDGE_RETRIEVAL_3/rag_pipeline_and_chunking.ipynb


### Maps to exam bullets:
- Select an appropriate chunking strategy given document structure, embedding model context
  length, and query types.

## Setup
A `.env` file with `DATABRICKS_HOST` and `DATABRICKS_TOKEN`

In [ ]:
import os
from dotenv import load_dotenv
import sys
sys.path.append(os.path.abspath(".."))
from _utils.sql_utils import sql_call

load_dotenv()

assert os.getenv("DATABRICKS_HOST"), "DATABRICKS_HOST not set -- check your .env file"
assert os.getenv("DATABRICKS_TOKEN"), "DATABRICKS_TOKEN not set -- check your .env file"
assert os.getenv("SQL_WAREHOUSE_ID"), "SQL_WAREHOUSE_ID not set -- check your .env file"

## Imports and config

In [ ]:
import numpy as np
import mlflow
from databricks_langchain import ChatDatabricks, DatabricksEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage

CATALOG, SCHEMA = "demo", "demo"
EMBEDDING_ENDPOINT = "databricks-gte-large-en"
LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct" #"claude_45" Claude is reliable when you need tools

embeddings = DatabricksEmbeddings(endpoint=EMBEDDING_ENDPOINT)
llm = ChatDatabricks(endpoint=LLM_ENDPOINT)

# TODO: change "oliver@mlops-media.com" to your own workspace user email
mlflow.set_experiment("/Users/oliver@mlops-media.com/rag_pipeline_and_chunking")

---
## 1 - Chunking strategy decision function

Source: [AI Search retrieval quality guide, Step 4](https://docs.databricks.com/aws/en/ai-search/retrieval-quality#step-4-improve-data-preparation)

The guide treats chunk size as Step 4 (lowest priority) and explicitly says "instead of
over-optimizing chunk size, focus on: metadata extraction, high-quality parsing, semantic
metadata." Three starting points are documented for experimentation :

- `small_chunks`   256  / Better for precise fact retrieval 
- `medium_chunks`  512  / Balanced approach 
- `large_chunks`   1024 / More context per chunk

Two **documented** advanced approaches are parent/child chunking and semantic chunking. 


In [ ]:
# --- Starting points for experimentation (verbatim from the guide) ---
small_chunks  = 256   # Better for precise fact retrieval
medium_chunks = 512   # Balanced approach
large_chunks  = 1024  # More context per chunk

# Key trade-offs documented in the guide:
# - Smaller chunks: better localization of specific information, but may lose context.
# - Larger chunks: more context preserved, but harder to pinpoint relevant information.
# - Context limits: must fit within LLM context window when retrieving multiple chunks.

def create_chunks(text, size_chars):
    """Minimal fixed-size splitter -- production code would use RecursiveCharacterTextSplitter."""
    words = text.split()
    avg_chars_per_word = 5
    size_words = max(1, size_chars // avg_chars_per_word)
    return [" ".join(words[i:i + size_words]) for i in range(0, len(words), size_words)]


In [ ]:
# return policy description
policy_doc = (
    "Our return policy allows items to be returned within 30 days of delivery for a full refund, "
    "provided the item is unused and in its original packaging. Refunds are issued to the original "
    "payment method within 5-7 business days of us receiving the returned item. "
    "All electronics carry a 1-year manufacturer warranty covering defects in materials and "
    "workmanship. The warranty does not cover accidental damage, water damage, or normal wear "
    "and tear. To file a warranty claim, contact support with your order ID."
)

In [ ]:
# --- Parent-child (small-to-big) chunking -- exact pattern from the guide ---
# Source: https://docs.databricks.com/aws/en/ai-search/retrieval-quality#chunking-strategy
# See also: https://python.langchain.com/docs/how_to/parent_document_retriever/

# Record child and parent chunks in your source table (pattern from the guide)
parent_child_rows = []
for parent_chunk in create_chunks(policy_doc, size_chars=2048):  # Large for context
    for child_chunk in create_chunks(parent_chunk, size_chars=256):  # Small for precision
        parent_child_rows.append({"text": child_chunk, "parent_text": parent_chunk})

print(f"Parent/child pairs produced: {len(parent_child_rows)}")
for row in parent_child_rows:
    print(f"  child ({len(row['text'])} chars): {row['text'][:40]}...{row['text'][-40:]}")
print(f"  parent({len(row['parent_text'])} chars): {row['parent_text'][:40]}...{row['parent_text'][-40:]}")
print()

# In production: write parent_child_rows to a UC Delta table, then create a Delta Sync Index
# with embedding_source_column="text". At query time, retrieve "text" + "parent_text" columns:
#
#   results = index.similarity_search(
#       query_text="Is attention all you need?",
#       num_results=10,
#       columns=["text", "parent_text"]   # search children, return parents
#   )

---
## 2 - Parsing strategy: recursive splitting vs `ai_parse_document` 

Section 1 decides *how big* a chunk should be. 
This section decides *what a chunk is allowed to contain* -- and that's where source **format** matters more than any size parameter. 
We generate two small PDFs, parse both with the real `ai parse document` SQL function, then build two chunk sets from the exact same parsed text: 
a generic recursive character split, 
and a structure-aware split that respects the elements `ai_parse_document` detected (so a table is never cut in half).


In [ ]:
import os
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.platypus import SimpleDocTemplate, Paragraph, Table, TableStyle, Spacer

LOCAL_DIR = "/tmp/rag_source_docs"
os.makedirs(LOCAL_DIR, exist_ok=True)
styles = getSampleStyleSheet()

In [ ]:
# Doc A: pure narrative, no table -- same return/warranty text used in section 3
doc_a_path = f"{LOCAL_DIR}/return_policy_context.pdf"
SimpleDocTemplate(doc_a_path, pagesize=letter).build([
    Paragraph("Returns & Warranty Policy", styles["Title"]),
    Spacer(1, 12),
    Paragraph(
        "Our return policy allows items to be returned within 30 days of delivery for a full "
        "refund, provided the item is unused and in its original packaging. Refunds are issued "
        "to the original payment method within 5-7 business days of us receiving the returned item.",
        styles["BodyText"]),
    Spacer(1, 12),
    Paragraph(
        "All electronics carry a 1-year manufacturer warranty covering defects in materials and "
        "workmanship. The warranty does not cover accidental damage, water damage, or normal wear "
        "and tear. To file a warranty claim, contact support with your order ID and a description "
        "of the issue.",
        styles["BodyText"]),
])

In [ ]:
# Doc B: narrative intro + an actual table -- the case ai_parse_document is built for.
# Reuses the same zone A/B/C shipping data as notebook 3, on purpose.
table_data = [
    ["Zone", "Standard delivery", "Express delivery", "Customs surcharge"],
    ["A (domestic)", "2-3 days, $4.99", "1 day, $14.99", "$0.00"],
    ["B (nearby intl.)", "5-7 days, $9.99", "2-3 days, $24.99", "$6.00"],
    ["C (remote intl.)", "10-14 days, $19.99", "Not available", "$15.00"],
]
table = Table(table_data, hAlign="LEFT")
table.setStyle(TableStyle([
    ("BACKGROUND", (0, 0), (-1, 0), "#0B3D5C"),
    ("TEXTCOLOR", (0, 0), (-1, 0), "#FFFFFF"),
    ("GRID", (0, 0), (-1, -1), 0.5, "#333333"),
    ("FONTSIZE", (0, 0), (-1, -1), 9),
]))
doc_b_path = f"{LOCAL_DIR}/shipping_rates_context.pdf"
SimpleDocTemplate(doc_b_path, pagesize=letter).build([
    Paragraph("Shipping Rates by Zone", styles["Title"]),
    Spacer(1, 12),
    Paragraph(
        "The table below summarizes delivery time and cost by shipping zone. Express delivery "
        "is not available for remote international destinations.",
        styles["BodyText"]),
    Spacer(1, 12),
    table,
])

In [ ]:
print(f"Generated {doc_a_path} and {doc_b_path}")

In [ ]:
from databricks.sdk import WorkspaceClient

DATABRICKS_CLIENT = WorkspaceClient()
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data/pdf_context"

# TODO: create the volume once if it does not exist yet:
#   CREATE VOLUME IF NOT EXISTS demo.demo.rag_docs
remote_pdf = []
for local_path in [doc_a_path, doc_b_path]:
    remote_path = f"{VOLUME_PATH}/{os.path.basename(local_path)}"
    with open(local_path, "rb") as f:
        DATABRICKS_CLIENT.files.upload(remote_path, f, overwrite=True)
        remote_pdf.append(remote_path)
    print(f"Uploaded {remote_path}")

In [ ]:
import json

def parse_document(volume_path):
    """Calls the real ai_parse_document SQL function against one file in a UC Volume."""
    rows = sql_call(
        databricks_sdk_client=DATABRICKS_CLIENT,
        statement=f"""
        SELECT to_json(ai_parse_document(content, map('version', '2.0'))) AS parsed
        FROM READ_FILES('{volume_path}', format => 'binaryFile')
        """,
        warehouse_id=os.getenv("SQL_WAREHOUSE_ID"),
    )
    return json.loads(rows[0]["parsed"])

parsed_docs = []
for pdf_path in remote_pdf: 
    parsed = parse_document(pdf_path)

    elements = parsed["document"]["elements"]
    print(f"{pdf_path} -> {len(elements)} elements: {[e['type'] for e in elements]}")
    parsed_docs.append(elements)

### Baseline: recursive character splitting on the flattened text

The most common "default" RAG ingestion: concatenate everything to one string and split on
generic separators, with no awareness of where a table starts or ends.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def flatten_elements(elements):
    return "\n\n".join(e["content"] for e in elements if e.get("content"))

splitter = RecursiveCharacterTextSplitter(chunk_size=220, chunk_overlap=20, separators=["\n\n", "\n", ". ", " "])

type_docs =["narrative_policy", "shipping_rates"] 

recursive_chunks = []
for doc_id, elements in zip(type_docs, parsed_docs):
    flat_text = flatten_elements(elements)
    for i, piece in enumerate(splitter.split_text(flat_text)):
        recursive_chunks.append({"doc_id": doc_id, "chunk_id": f"{doc_id}_recursive_{i}", "text": piece})

print(f"Recursive splitting produced {len(recursive_chunks)} chunks")
for c in recursive_chunks:
    if c["doc_id"] == "shipping_rates":
        print(f"  [{c['chunk_id']}] {c['text'][:120]!r}")

### Structure-aware: chunk by `ai_parse_document` element

A table element is never split -- it becomes exactly one chunk, however long. Plain-text
elements (paragraphs, headers) are still merged up to the same size budget so we are not just
comparing "many small chunks" to "few large chunks."


In [ ]:
def chunk_by_element(doc_id, elements, max_chars=220):
    chunks, buffer = [], ""
    for e in elements:
        content = e.get("content", "")
        if not content:
            continue
        if e["type"] == "table":
            if buffer:
                chunks.append(buffer)
                buffer = ""
            chunks.append(content)  # tables are always their own chunk, never split
            continue
        if len(buffer) + len(content) <= max_chars:
            buffer = (buffer + "\n\n" + content).strip()
        else:
            if buffer:
                chunks.append(buffer)
            buffer = content
    if buffer:
        chunks.append(buffer)
    return [{"doc_id": doc_id, "chunk_id": f"{doc_id}_struct_{i}", "text": t} for i, t in enumerate(chunks)]


In [ ]:
structured_chunks = [c for doc_type, elements in zip(type_docs, parsed_docs) for c in chunk_by_element(doc_type, elements)]
print(f"Structure-aware chunking produced {len(structured_chunks)} chunks")
for c in structured_chunks:
    if c["doc_id"] == "shipping_rates":
        print(f"  [{c['chunk_id']}] {c['text'][:160]!r}")

# --- Going to production ----------------------------------------------------
# Skip the hand-rolled chunk_by_element() above and let Databricks do it natively:
#
#   SELECT ai_prep_search(ai_parse_document(content, map('version', '2.0'))) AS prepped
#   FROM READ_FILES('{VOLUME_PATH}', format => 'binaryFile')
#
# ai_prep_search returns chunk_to_retrieve / chunk_to_embed pairs that already respect
# element boundaries (tables stay intact) and prepend section/document context to each chunk --
# this is the recommended production path, shown here as SQL only to keep the comparison above
# transparent and inspectable.
# -----------------------------------------------------------------------------

### Compare retrieval on a narrative fact vs a table fact


In [ ]:
def build_chunk_index(chunks):
    vectors = embeddings.embed_documents([c["text"] for c in chunks])
    return [{**c, "vector": np.array(v)} for c, v in zip(chunks, vectors)]

def retrieve_from(query, index, k=2):
    qvec = np.array(embeddings.embed_query(query))
    scored = sorted(
        index,
        key=lambda it: float(np.dot(qvec, it["vector"]) / ((np.linalg.norm(qvec) * np.linalg.norm(it["vector"])) or 1e-9)),
        reverse=True,
    )
    return scored[:k]

recursive_index = build_chunk_index(recursive_chunks)
structured_index = build_chunk_index(structured_chunks)

parse_eval_queries = [
    {"query": "How long do I have to return an item?", "must_contain": ["30 days"], "kind": "narrative_fact"},
    {"query": "What is the express delivery cost for zone B?", "must_contain": ["24.99"], "kind": "table_fact"},
]

def evaluate_parsing(index, label):
    print(f"--- {label} ---")
    for q in parse_eval_queries:
        hits = retrieve_from(q["query"], index, k=2)
        retrieved_text = " ".join(h["text"] for h in hits).lower()
        hit_ok = all(term.lower() in retrieved_text for term in q["must_contain"])
        status = "HIT " if hit_ok else "MISS"
        print(f"  [{q['kind']:>14}] {status} -- {q['query']}")

evaluate_parsing(recursive_index, "recursive splitting (flat text)")
print()
evaluate_parsing(structured_index, "structure-aware (ai_parse_document elements)")

### Diagnosis and a format-driven decision function

- Narrative-only fact : 
both strategies usually succeed -- plain prose chunks cleanly under either approach.
- Table-embedded fact : 
recursive splitting on flattened text can separate a table row from its header row, or cut the HTML mid-row -- the retrieved
chunk no longer carries a complete, attributable fact. ai_parse_document keeps each table as one atomic element, so the
structure-aware chunk (or ai_prep_search output) always contains the full row AND its header, regardless of chunk-size tuning.

## 3 - The full pre-inference RAG pipeline, end to end

`parse -> chunk -> embed -> index -> retrieve -> inject into agent context`. 
The corpus here stands in for a UC Volume of governed documents (product/shipping policy docs); 
in production the parse + chunk steps would run as a Delta Live Table pipeline writing into a UC 
table that a real Vector Search Delta Sync index syncs against.

In [ ]:
policy_docs = [
    {"doc_id": "doc_returns", "text":
        "Our return policy allows items to be returned within 30 days of delivery for a full refund, "
        "provided the item is unused and in its original packaging. Refunds are issued to the original "
        "payment method within 5-7 business days of us receiving the returned item."},
    {"doc_id": "doc_warranty", "text":
        "All electronics carry a 1-year manufacturer warranty covering defects in materials and workmanship. "
        "The warranty does not cover accidental damage, water damage, or normal wear and tear. To file a "
        "warranty claim, contact support with your order ID and a description of the issue."},
]

def chunk_document(doc, chunk_size_words=40, overlap_words=8):
    words = doc["text"].split()
    chunks, start = [], 0
    while start < len(words):
        end = min(start + chunk_size_words, len(words))
        chunks.append({"doc_id": doc["doc_id"], "chunk_id": f"{doc['doc_id']}_{start}", "text": " ".join(words[start:end])})
        start += chunk_size_words - overlap_words
    return chunks



In [ ]:
strategy = {'chunk_size': 256, 'overlap': 32, 'strategy': 'parent/child: embed small child chunks, return the larger parent chunk on a hit'}
chunk_words = max(20, strategy["chunk_size"] // 8)   # rough token->word downscale for this toy corpus
all_chunks = [c for doc in policy_docs for c in chunk_document(doc, chunk_size_words=chunk_words, overlap_words=4)]
print(f"Chunking strategy used: {strategy}")
print(f"Produced {len(all_chunks)} chunks from {len(policy_docs)} documents")

In [ ]:
chunk_vectors = embeddings.embed_documents([c["text"] for c in all_chunks])
policy_index = [{**c, "vector": np.array(v)} for c, v in zip(all_chunks, chunk_vectors)]

def retrieve(query, index, k=2):
    qvec = np.array(embeddings.embed_query(query))
    scored = sorted(
        index,
        key=lambda it: float(np.dot(qvec, it["vector"]) / ((np.linalg.norm(qvec) * np.linalg.norm(it["vector"])) or 1e-9)),
        reverse=True,
    )
    return scored[:k]

def ask_pre_inference(question, index, k=2):
    hits = retrieve(question, index, k=k)
    context = "\n".join(f"- {h['text']}" for h in hits)
    messages = [
        SystemMessage(content=f"Answer using only this context:\n{context}\nIf the context does not contain the answer, say so."),
        HumanMessage(content=question),
    ]
    return llm.invoke(messages).content, hits



In [ ]:
answer, hits = ask_pre_inference("How long do I have to return an item?", policy_index)
print("Answer:", answer)
print("Knowledge used :",  [h['chunk_id'] for h in hits])
for knowledge in [h['text'] for h in hits] :
    print(f"Knowledge : ", knowledge)